In [0]:
from utils.logger import get_logger
from etl_config.gold_fact_config import FACT_CONFIG, CATALOG, GOLD_SCHEMA

logger = get_logger("gold_fact_setup")

In [0]:
for fact_name, cfg in FACT_CONFIG.items():
    full_table_name = cfg.target_table

    col_ddl = ",\n        ".join(
        f"{c.name} {c.sql_type}" + ("" if c.nullable else " not null")
        for c in cfg.columns
    )

    spark.sql(f"""
        create table if not exists {full_table_name} (
            {col_ddl}
        )
        using delta
        comment '{cfg.table_description}'
    """)
    logger.info(f"Fact table created/verified: {full_table_name}")

    spark.sql(f"""
        alter table {full_table_name}
        set tblproperties (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true',
            'description' = '{cfg.table_description}'
        )
    """)

    for c in cfg.columns:
        if c.comment:
            spark.sql(
                f"comment on column {full_table_name}.{c.name} is :comment",
                args={"comment": c.comment}
            )

    logger.info(f"Metadata applied for: {full_table_name}")

logger.info("Gold fact setup complete.")